# sdcguard in 2 minutes

Catch a GPU that silently computes wrong numbers, from inside the training run.

On Colab, pick a GPU first: **Runtime > Change runtime type > T4 GPU**. It also runs on CPU, just slower.


In [ ]:
!pip install -q sdcguard

## 1. Self-test
Checks that sdcguard sees and checks this machine's matrix multiplies, with no false alarms, and catches injected faults. It should end with `ALL PASS`.


In [ ]:
!python -m sdcguard.selftest

## 2. The demo
A small network trains. From step 45 a simulated bad device corrupts one layer's outputs with the fault patterns NVIDIA found in a gate-level fault-injection study of a data-center GPU. Watch sdcguard catch it, name the layer, and roll back. The corruption is simulated in software.


In [ ]:
!python -m sdcguard.demo

## 3. In your own training loop
Wrap the forward and backward pass of every Nth step. Nothing else changes.


In [ ]:
from contextlib import nullcontext

import torch
import torch.nn as nn
import torch.nn.functional as F

from sdcguard import SDCGuard

device = "cuda" if torch.cuda.is_available() else "cpu"
# sdcguard checks bf16/fp16 matmuls. GPUs without native bf16 (like Colab's free T4) use fp16.
bf16_ok = device == "cpu" or torch.cuda.is_bf16_supported(including_emulation=False)
dtype = torch.bfloat16 if bf16_ok else torch.float16
model = nn.Sequential(nn.Linear(512, 2048), nn.GELU(), nn.Linear(2048, 512)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
guard = SDCGuard().watch(model)          # watch() is optional: it names layers in alerts

for step in range(200):
    x = torch.randn(64, 512, device=device)
    checked = step % 20 == 0             # every 20th step: about +5.6% time on GPT-2 small (H100)
    with guard.at_step(step) if checked else nullcontext():
        with torch.autocast(device, dtype=dtype):
            loss = F.mse_loss(model(x).float(), x)
        loss.backward()
    opt.step()
    opt.zero_grad(set_to_none=True)
    if guard.log.events:                 # an alert: stop, find the GPU, roll back
        print(guard.log.events[0].to_json())
        break

print(f"checked {guard.stats['checked']} matmuls, {guard.stats['alerts']} alerts")


## 4. Check this GPU, and put it on the public board

`sdcguard check` trains and serves a small model with every matmul checked, runs matmuls in every precision the GPU has against a float64 reference, tests most of the memory and measures speed. About 3 minutes here. `--share` prints a link: open it to post this GPU's result card to the [public board](https://hari-kancharla.github.io/sdcguard/board/). Nothing is sent until you press Submit there.

In [ ]:
!sdcguard check --minutes 3 --share